In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

server = os.getenv('SQL_SERVER')
database = os.getenv('SQL_DATABASE')

connection_string = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(connection_string)

try:
    with engine.connect() as conn:
        print(f"Conectado com sucesso ao banco: {database}")
except Exception as e:
    print(f"Erro na conexão: {e}")

In [ ]:
# Parâmetros
TOTAL_CLIENTES = 5000
TOTAL_ANALISTAS = 20
TEMPO_ANALISE_MINUTOS = 15
TAXA_DESISTENCIA_MANUAL = 0.60
TAXA_DESISTENCIA_REVISAO = 0.15
PERDA_INADIMPLENCIA = 0.70
SLA_HORAS = 24

# Carregar dados do banco
df_decisoes = pd.read_sql("SELECT * FROM decisoes_credito", engine)

aprovados = len(df_decisoes[df_decisoes['decisao'] == 'APROVADO'])
negados = len(df_decisoes[df_decisoes['decisao'] == 'NEGADO'])
revisao = len(df_decisoes[df_decisoes['decisao'] == 'REVISAO'])
valor_medio = df_decisoes['valor_solicitado'].mean()

print(f"Aprovados: {aprovados}")
print(f"Negados: {negados}")
print(f"Revisão: {revisao}")
print(f"Valor médio solicitado: R${valor_medio:.2f}")

In [ ]:
# Cenário A: Processo manual (sem o sistema)
print("=== CENÁRIO A: PROCESSO MANUAL ===\n")

# Tempo total
tempo_total_horas = (TOTAL_CLIENTES * TEMPO_ANALISE_MINUTOS) / 60
tempo_por_analista = tempo_total_horas / TOTAL_ANALISTAS
tempo_total_dias = tempo_por_analista / 8

# Analistas necessários para cumprir SLA
analistas_necessarios = int(tempo_total_horas / SLA_HORAS) + 1
analistas_faltando = max(0, analistas_necessarios - TOTAL_ANALISTAS)

# Tempo real disponível por cliente
minutos_disponiveis_por_cliente = (TOTAL_ANALISTAS * SLA_HORAS * 60) / TOTAL_CLIENTES

# Clientes perdidos por tempo de resposta
horas_disponiveis = TOTAL_ANALISTAS * SLA_HORAS
clientes_no_prazo = int((horas_disponiveis * 60) / TEMPO_ANALISE_MINUTOS)
clientes_perdidos_a = max(0, TOTAL_CLIENTES - clientes_no_prazo)
credito_nao_convertido_a = clientes_perdidos_a * TAXA_DESISTENCIA_MANUAL * valor_medio

# Perda por inadimplência
taxa_inadimplencia_manual = 0.35
inadimplentes_manual = int(TOTAL_CLIENTES * taxa_inadimplencia_manual)
perda_inadimplencia_manual = inadimplentes_manual * valor_medio * PERDA_INADIMPLENCIA

print(f"Tempo total para analisar: {tempo_total_horas:.0f} horas ({tempo_total_dias:.0f} dias úteis por analista)")
print(f"Tempo por analista: {tempo_por_analista:.0f} horas")
print(f"\nAnalistas necessários para cumprir SLA: {analistas_necessarios}")
print(f"Analistas disponíveis: {TOTAL_ANALISTAS}")
print(f"Analistas faltando: {analistas_faltando}")
print(f"\nTempo disponível por análise para cumprir SLA: {minutos_disponiveis_por_cliente:.1f} minutos")
print(f"Tempo ideal por análise: {TEMPO_ANALISE_MINUTOS} minutos")
print(f"Analistas teriam que reduzir tempo de análise em {((TEMPO_ANALISE_MINUTOS - minutos_disponiveis_por_cliente) / TEMPO_ANALISE_MINUTOS * 100):.0f}% — alto risco de erro humano")
print(f"\nClientes perdidos por tempo de resposta: {clientes_perdidos_a}")
print(f"Crédito não convertido: R${credito_nao_convertido_a:,.2f}")
print(f"\nTaxa de inadimplência: 35%")
print(f"Motivo: analistas com {minutos_disponiveis_por_cliente:.1f} min por cliente cometem mais erros de avaliação")
print(f"Inadimplentes aprovados: {inadimplentes_manual}")
print(f"Perda por inadimplência: R${perda_inadimplencia_manual:,.2f}")
print(f"\nTotal de perdas: R${credito_nao_convertido_a + perda_inadimplencia_manual:,.2f}")

In [ ]:
# Cenário B: Com o sistema automatizado
print("=== CENÁRIO B: COM O SISTEMA AUTOMATIZADO ===\n")

# Clientes analisados
clientes_revisao = revisao  # 318 clientes

# Tempo total
tempo_total_horas_b = (clientes_revisao * TEMPO_ANALISE_MINUTOS) / 60
tempo_por_analista_b = tempo_total_horas_b / TOTAL_ANALISTAS

# Analistas necessários
analistas_necessarios_b = max(1, int(tempo_total_horas_b / SLA_HORAS) + 1)

# Clientes perdidos por tempo de resposta
TAXA_DESISTENCIA_REVISAO = 0.15
clientes_perdidos_b = int(clientes_revisao * TAXA_DESISTENCIA_REVISAO)
credito_nao_convertido_b = clientes_perdidos_b * valor_medio

# Perda por inadimplência
# 10% pois IA já filtrou casos críticos
inadimplentes_b = int(aprovados * 0.10)
perda_inadimplencia_b = inadimplentes_b * valor_medio * PERDA_INADIMPLENCIA

print(f"Clientes analisados pela IA automaticamente: {aprovados + negados}")
print(f"Clientes que precisam de análise manual: {clientes_revisao}")
print(f"\nTempo total para analisar revisões: {tempo_total_horas_b:.1f} horas")
print(f"Tempo por analista: {tempo_por_analista_b:.1f} horas")
print(f"\nAnalistas necessários para cumprir SLA: {analistas_necessarios_b}")
print(f"\nClientes perdidos por tempo de resposta: {clientes_perdidos_b} ({TAXA_DESISTENCIA_REVISAO:.0%})")
print(f"Crédito não convertido: R${credito_nao_convertido_b:,.2f}")
print(f"\nTaxa de inadimplência: 10%")
print(f"Inadimplentes aprovados: {inadimplentes_b}")
print(f"Perda por inadimplência: R${perda_inadimplencia_b:,.2f}")
print(f"\nTotal de perdas: R${credito_nao_convertido_b + perda_inadimplencia_b:,.2f}")

In [ ]:
# Comparação dos cenários
print("=== COMPARAÇÃO: MANUAL vs AUTOMATIZADO ===\n")

# Tempo
reducao_tempo = ((tempo_total_horas - tempo_total_horas_b) / tempo_total_horas) * 100
print(f"TEMPO DE ANÁLISE:")
print(f"  Manual:       {tempo_total_horas:.0f} horas")
print(f"  Automatizado: {tempo_total_horas_b:.1f} horas")
print(f"  Redução:      {reducao_tempo:.0f}%")

# Analistas
print(f"\nANALISTAS:")
print(f"  Necessários no processo manual:       {analistas_necessarios}")
print(f"  Necessários com o sistema:            {analistas_necessarios_b}")
print(f"  Analistas liberados para outras atividades: {analistas_necessarios - analistas_necessarios_b}")

# Clientes perdidos
reducao_clientes_perdidos = ((clientes_perdidos_a - clientes_perdidos_b) / clientes_perdidos_a) * 100
print(f"\nCLIENTES PERDIDOS POR TEMPO DE RESPOSTA:")
print(f"  Manual:       {clientes_perdidos_a}")
print(f"  Automatizado: {clientes_perdidos_b}")
print(f"  Redução:      {reducao_clientes_perdidos:.0f}%")

# Crédito não convertido
reducao_credito = ((credito_nao_convertido_a - credito_nao_convertido_b) / credito_nao_convertido_a) * 100
print(f"\nCRÉDITO NÃO CONVERTIDO:")
print(f"  Manual:       R${credito_nao_convertido_a:,.2f}")
print(f"  Automatizado: R${credito_nao_convertido_b:,.2f}")
print(f"  Redução:      {reducao_credito:.0f}%")

# Inadimplência
print(f"\nPERDA POR INADIMPLÊNCIA:")
print(f"  Manual:       R${perda_inadimplencia_manual:,.2f} (35%)")
print(f"  Automatizado: R${perda_inadimplencia_b:,.2f} (10%)")
print(f"  Redução:      R${perda_inadimplencia_manual - perda_inadimplencia_b:,.2f}")

# Total
total_a = credito_nao_convertido_a + perda_inadimplencia_manual
total_b = credito_nao_convertido_b + perda_inadimplencia_b
reducao_total = ((total_a - total_b) / total_a) * 100
print(f"\nTOTAL DE PERDAS:")
print(f"  Manual:       R${total_a:,.2f}")
print(f"  Automatizado: R${total_b:,.2f}")
print(f"  Redução:      {reducao_total:.0f}% — R${total_a - total_b:,.2f} preservados")

In [ ]:
# Salvar resultados no banco
resultados_simulacao = pd.DataFrame([{
    'cenario': 'Manual',
    'tempo_analise_horas': tempo_total_horas,
    'analistas_necessarios': analistas_necessarios,
    'clientes_perdidos': clientes_perdidos_a,
    'credito_nao_convertido': credito_nao_convertido_a,
    'perda_inadimplencia': perda_inadimplencia_manual,
    'total_perdas': total_a
}, {
    'cenario': 'Automatizado',
    'tempo_analise_horas': tempo_total_horas_b,
    'analistas_necessarios': analistas_necessarios_b,
    'clientes_perdidos': clientes_perdidos_b,
    'credito_nao_convertido': credito_nao_convertido_b,
    'perda_inadimplencia': perda_inadimplencia_b,
    'total_perdas': total_b
}])

try:
    resultados_simulacao.to_sql('simulacao_operacional', engine, if_exists='replace', index=False)
    print("Resultados salvos com sucesso!")
    print(resultados_simulacao.to_string())
except Exception as e:
    print(f"Erro ao salvar: {e}")